In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os

# Set up mixed precision removed to prevent float32 casting issues

# Mapping PlantDoc classes to your original 38 classes (same as in your notebook)
label_mapping = {
    "Apple Scab Leaf": 0,
    "Apple leaf": 3,
    "Apple rust leaf": 2,
    "Bell_pepper leaf": 19,
    "Bell_pepper leaf spot": 18,
    "Blueberry leaf": 4,
    "Cherry leaf": 6,
    "Corn Gray leaf spot": 7,
    "Corn leaf blight": 9,
    "Corn rust leaf": 8,
    "Peach leaf": 17,
    "Potato leaf early blight": 20,
    "Potato leaf late blight": 21,
    "Raspberry leaf": 23,
    "Soyabean leaf": 24,
    "Squash Powdery mildew leaf": 25,
    "Strawberry leaf": 27,
    "Tomato Early blight leaf": 29,
    "Tomato Septoria leaf spot": 32,
    "Tomato leaf": 37,
    "Tomato leaf bacterial spot": 28,
    "Tomato leaf late blight": 30,
    "Tomato leaf mosaic virus": 36,
    "Tomato leaf yellow virus": 35,
    "Tomato mold leaf": 31,
    "grape leaf": 14,
    "grape leaf black rot": 11
}

# Reverse mapping to get class name from index
idx_to_class = {v: k for k, v in label_mapping.items()}

class MultiViewPredictor:
    def __init__(self, model_path='test_model.keras', img_size=(224, 224)):
        print(f"Loading model from {model_path}...")
        self.model = keras.models.load_model(model_path)
        self.img_size = img_size
        print("Model loaded successfully.")

    def preprocess_image(self, path):
        """Loads and preprocesses a single image."""
        if not os.path.exists(path):
            raise FileNotFoundError(f"Image not found: {path}")
        
        image = tf.io.read_file(path)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, self.img_size)
        # Convert to float32 before model processing (matches training pipeline)
        image = tf.cast(image, tf.float32)
        # Add batch dimension
        image = tf.expand_dims(image, 0)
        return image

    def predict_multi_view(self, image_paths):
        """
        Predicts the disease based on multiple images of the SAME plant.
        Combines probabilities (Soft Voting) to get a robust prediction.
        """
        if not image_paths:
            raise ValueError("Please provide at least one image path.")
        
        print(f"Processing {len(image_paths)} images...")
        
        all_probs = []
        for path in image_paths:
            img = self.preprocess_image(path)
            # Get probability distribution for this image
            # The model outputs raw probabilities (softmax)
            probs = self.model.predict(img, verbose=0)[0]
            
            # Cast back to float32 if mixed_precision returned float16
            probs = np.array(probs, dtype=np.float32)
            all_probs.append(probs)
            
            # Print individual prediction for insight
            pred_idx = np.argmax(probs)
            confidence = probs[pred_idx] * 100
            class_name = idx_to_class.get(pred_idx, f"Unknown (ID: {pred_idx})")
            print(f" - {os.path.basename(path)}: {class_name} ({confidence:.2f}%)")
            
        # Stack probabilities and average them (Soft Voting)
        all_probs_stacked = np.stack(all_probs)
        avg_probs = np.mean(all_probs_stacked, axis=0)
        
        # Get final combined prediction
        final_idx = np.argmax(avg_probs)
        final_confidence = avg_probs[final_idx] * 100
        final_class = idx_to_class.get(final_idx, f"Unknown (ID: {final_idx})")
        
        print("\n=== COMBINED MULTI-VIEW PREDICTION ===")
        print(f"Predicted Disease : {final_class}")
        print(f"Overall Confidence: {final_confidence:.2f}%")
        
        return final_class, final_confidence, avg_probs

if __name__ == "__main__":
    # Example usage:
    predictor = MultiViewPredictor(model_path='test_model.keras')
    leaf_photos = [
         "C:\\plant_disease_dataset\\New Plant Diseases Dataset(Augmented)\\New Plant Diseases Dataset(Augmented)\\train\\Apple___Apple_scab\\0a5e9323-dbad-432d-ac58-d291718345d9___FREC_Scab 3417_90deg.JPG",
         "C:\\plant_disease_dataset\\New Plant Diseases Dataset(Augmented)\\New Plant Diseases Dataset(Augmented)\\train\\Apple___Apple_scab\\0b170906-9436-4c0d-84c1-c396ad9d909b___FREC_Scab 3101.JPG",
         "C:\\Users\\krish\\OneDrive\\Desktop\\plant\\test_dataset\\PlantDoc-Dataset\\test\\Apple Scab Leaf\\Apple Scab Leaf (1).jpg"
     ]
    # 
    predicted_class, confidence, _ = predictor.predict_multi_view(leaf_photos)
    print("Multi-View Predictor initialized. To use, import this module or uncomment the example block.")


Loading model from test_model.keras...
Model loaded successfully.
Processing 3 images...
 - 0a5e9323-dbad-432d-ac58-d291718345d9___FREC_Scab 3417_90deg.JPG: Apple Scab Leaf (82.94%)
 - 0b170906-9436-4c0d-84c1-c396ad9d909b___FREC_Scab 3101.JPG: Apple Scab Leaf (94.06%)
 - Apple Scab Leaf (1).jpg: Apple rust leaf (65.17%)

=== COMBINED MULTI-VIEW PREDICTION ===
Predicted Disease : Apple Scab Leaf
Overall Confidence: 68.40%
Multi-View Predictor initialized. To use, import this module or uncomment the example block.
